In [2]:
import torch


# g(x) = max(0,x) + min(0,x)
x2 = torch.tensor(0.0, requires_grad=True)


# For clarity, also check at nearby points
for val in [-1.0, 0.0, 1.0]:
    x = torch.tensor(val, requires_grad=True)
    g = torch.sqrt(x)
    g.backward()
    print(f"dg/dx at x={val}: {x.grad.item()}")

dg/dx at x=-1.0: nan
dg/dx at x=0.0: inf
dg/dx at x=1.0: 0.5


In [3]:
import torch

class SafeSqrt(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.sqrt(x)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        # standard gradient
        grad_input = grad_output * 0.5 / torch.sqrt(x)
        # hardcode gradient at 0.0
        grad_input = torch.where(x == 0, torch.tensor(100000., device=x.device, dtype=x.dtype), grad_input)
        return grad_input

# Test at -1.0, 0.0, 1.0
for val in [-1.0, 0.0, 1.0]:
    x = torch.tensor(val, requires_grad=True)
    try:
        g = SafeSqrt.apply(x)
        g.backward()
        print(f"dg/dx at x={val}: {x.grad.item()}")
    except RuntimeError as e:
        print(f"x={val} raised an error: {e}")


dg/dx at x=-1.0: nan
dg/dx at x=0.0: 0.0
dg/dx at x=1.0: 0.5


In [4]:
import torch

x2 = torch.tensor(0.0, requires_grad=True)
y2 = torch.tensor(0.0, requires_grad=True)

g = torch.atan2(y2, x2)
g.backward()

print(f"dg/dx at (0,0): {x2.grad.item()}")
print(f"dg/dy at (0,0): {y2.grad.item()}")


dg/dx at (0,0): nan
dg/dy at (0,0): nan


In [5]:
class SafeAtan2(torch.autograd.Function):
    @staticmethod
    def forward(ctx, y, x):
        ctx.save_for_backward(x, y)
        return torch.atan2(y, x)

    @staticmethod
    def backward(ctx, grad_output):
        x, y = ctx.saved_tensors
        denom = x**2 + y**2
        grad_x = -y / denom
        grad_y = x / denom
        # override gradient at (0,0) to 0
        mask = (x == 0) & (y == 0)
        #grad_x = torch.where(mask, torch.tensor(x., device=x.device, dtype=x.dtype), grad_x)
        grad_x = 0
        grad_y = torch.where(mask, torch.tensor(0., device=y.device, dtype=y.dtype), grad_y)
        return grad_y * grad_output, grad_x * grad_output

# Test
x2 = torch.tensor(0.0, requires_grad=True)
y2 = torch.tensor(0.0, requires_grad=True)
g = SafeAtan2.apply(y2, x2)
g.backward()
print(x2.grad, y2.grad)  # 0.0, 0.0


tensor(0.) tensor(0.)
